In [1]:
from utils import simulation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

### Simulations Parameter ##############
n = 1000
seed = 42
n_sim = 1000
B_1 = 200

jk_ab_calc = False
boot_var_calc = False
ijk_std_calc = False
train_models = False
B_RF = int(n*0.7  * 2)

########## Varying Parameters ##########
tau = 12

params_data_creation = { 'shape_weibull': 1, 'scale_weibull_base': 10000, 'rate_censoring': 0.01,
                        'b_bloodp': -0.405, 'b_diab': -0.4, 'b_age': -0.05, 'b_bmi': -0.01, 'b_kreat': -0.2, 
                        'n': n, 'seed': seed, 'tau': tau}

params_rf = {  'n_estimators':B_RF,                        
                'max_depth':3,
                'min_samples_split':5,
                'max_features': 'log2',
                'random_state':  seed,
                'bootstrap':     True,  }
X_erwartung = pd.DataFrame({'bmi': [25], 'blood_pressure': [0], 'kreatinkinase': [np.exp(5+1/2)], 'diabetes': [0], 'age': [50]})

######## Start Simulation   ########
with ProcessPoolExecutor() as executor:
    
    ### Array to store the results
    portion_events_after_cut_train = np.zeros(n_sim)
    portion_censored_after_cut_train = np.zeros(n_sim)
    portion_no_events_after_cut_train = np.zeros(n_sim)
    portion_events_after_cut_test = np.zeros(n_sim)
    portion_censored_after_cut_test = np.zeros(n_sim)
    portion_no_events_after_cut_test = np.zeros(n_sim)
    wb_mse_ipcw = np.zeros(n_sim)
    wb_cindex_ipcw = np.zeros(n_sim)
    wb_y_pred_X_point = np.zeros(n_sim)
    rf_mse_ipcw = np.zeros(n_sim)
    rf_y_pred_X_point = np.zeros(n_sim)
    ijk_var_pred_X_point = np.zeros(n_sim)
    bootstrap_std_pred_X_point = np.zeros(n_sim)

    futures = [
        executor.submit(
            simulation,
            seed=seed+i,
            tau=tau, 
            data_generation_weibull_parameters=params_data_creation,
            params_rf=params_rf, 
            X_pred_point=X_erwartung,
            B_first_level = B_1,
            boot_std_calc = boot_var_calc,
            ijk_std_calc = ijk_std_calc,
            train_models = train_models,
            jk_ab_calc = jk_ab_calc
        )
        for i in range(n_sim)
    ]

    for i, future in enumerate(tqdm(futures, desc="Simulations", unit="simulation")):
        _portion_events_after_cut_train, _portion_censored_after_cut_train, _portion_no_events_after_cut_train, \
         _portion_events_after_cut_test, _portion_censored_after_cut_test, _portion_no_events_after_cut_test, \
        _wb_mse_ipcw, _wb_cindex_ipcw, _wb_y_pred_X_point, \
        _rf_mse_ipcw, _rf_y_pred_X_point, _ijk_var_pred_X_point, _bootstrap_std_pred_X_point  = future.result()

        #Event-Stats Results
        portion_events_after_cut_train[i] = _portion_events_after_cut_train
        portion_censored_after_cut_train[i] = _portion_censored_after_cut_train
        portion_no_events_after_cut_train[i] = _portion_no_events_after_cut_train
        portion_events_after_cut_test[i] = _portion_events_after_cut_test
        portion_censored_after_cut_test[i] = _portion_censored_after_cut_test
        portion_no_events_after_cut_test[i] = _portion_no_events_after_cut_test
        
        #Evaluation Results
        wb_mse_ipcw[i] = _wb_mse_ipcw
        wb_cindex_ipcw[i] = _wb_cindex_ipcw
        rf_mse_ipcw[i] = _rf_mse_ipcw

        #Prediction Results
        wb_y_pred_X_point[i] = _wb_y_pred_X_point[0]
        rf_y_pred_X_point[i] = _rf_y_pred_X_point[0]

        # Standard Deviation Estimates
        ijk_var_pred_X_point[i]  = _ijk_var_pred_X_point
        bootstrap_std_pred_X_point[i] = _bootstrap_std_pred_X_point
def print_stats_data():

    print('Event-Stats Results:')
    print('Train:')
    print(f'Portion of events after cut:     {round(np.mean(portion_events_after_cut_train)*100,2)}%,   n={round(n*0.7*np.mean(portion_events_after_cut_train),0)}')
    print(f'Portion of no events after cut:  {round(np.mean(portion_no_events_after_cut_train)*100,2)}%,    n={round(n*0.7*np.mean(portion_no_events_after_cut_train),0)}')
    print(f'Portion of censored after cut:   {round(np.mean(portion_censored_after_cut_train)*100,2)}%,   n={round(n*0.7*np.mean(portion_censored_after_cut_train),0)}\n')
    print('Test:')
    print(f'Portion of events after cut:     {round(np.mean(portion_events_after_cut_test)*100,2)}%,   n={round(n*0.7*np.mean(portion_events_after_cut_test),0)}')
    print(f'Portion of no events after cut:  {round(np.mean(portion_no_events_after_cut_test)*100,2)}%,    n={round(n*0.7*np.mean(portion_no_events_after_cut_test),0)}')
    print(f'Portion of censored after cut:   {round(np.mean(portion_censored_after_cut_test)*100,2)}%,   n={round(n*0.7*np.mean(portion_censored_after_cut_test),0)}')
    print('\n')
print_stats_data()

Simulations: 100%|██████████| 1000/1000 [00:04<00:00, 231.08simulation/s]


Event-Stats Results:
Train:
Portion of events after cut:     82.01%,   n=574.0
Portion of no events after cut:  7.12%,    n=50.0
Portion of censored after cut:   10.87%,   n=76.0

Test:
Portion of events after cut:     81.93%,   n=574.0
Portion of no events after cut:  7.19%,    n=50.0
Portion of censored after cut:   10.87%,   n=76.0


